# Clase 192 — Bayes: priors, posterior, MCMC con PyMC

Lógica bayesiana: `posterior ∝ likelihood × prior`. Mostramos el caso conjugado **Beta-Binomial** de forma ejecutable con scipy, y el stack moderno **PyMC v5 + ArviZ** en código correcto (no se ejecuta aquí porque PyMC no está instalado en este entorno).

Requiere ejecutar el conjugado: `numpy`, `scipy`, `matplotlib`. Para los modelos MCMC: `pip install pymc arviz numpyro`.

## 1. Conjugado Beta-Binomial (ejecutable)

Con prior `Beta(1,1)` (uniforme) y datos binomiales, el posterior es `Beta(a+éxitos, b+fracasos)` — sin necesidad de MCMC. Calculamos el intervalo del 94 % con `scipy`.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# 100 visitas, 8 conversiones. Prior Beta(1,1)
visitas, conversiones = 100, 8
a0, b0 = 1, 1
a_post, b_post = a0 + conversiones, b0 + (visitas - conversiones)   # Beta(9, 93)
post = stats.beta(a_post, b_post)
lo, hi = post.ppf(0.03), post.ppf(0.97)    # intervalo central del 94%
print(f"Posterior = Beta({a_post}, {b_post})")
print(f"media posterior = {post.mean():.3f}")
print(f"intervalo 94% = ({lo:.3f}, {hi:.3f})")

grid = np.linspace(0, 0.25, 300)
plt.figure(figsize=(7, 4))
plt.plot(grid, stats.beta(a0, b0).pdf(grid), label="prior Beta(1,1)")
plt.plot(grid, post.pdf(grid), label=f"posterior Beta({a_post},{b_post})")
plt.title("Actualización bayesiana: prior -> posterior"); plt.legend()
plt.tight_layout(); plt.show()

## 2. Regresión lineal bayesiana con PyMC v5

Código moderno (PyTensor backend). NUTS muestrea el posterior; ArviZ diagnostica convergencia (`r_hat ≤ 1.01`, `ess_bulk ≥ 400`). **No se ejecuta aquí** (PyMC ausente).

In [ ]:
# Requiere: pip install pymc arviz  (NO se ejecuta en este entorno)
import numpy as np
import pymc as pm
import arviz as az

rng = np.random.default_rng(42)
x = rng.normal(0, 1, 200)
y = 1.0 + 2.5 * x + rng.normal(0, 1, 200)

with pm.Model() as modelo:
    alpha = pm.Normal("alpha", mu=0, sigma=10)          # priors débilmente informativos
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)
    mu = alpha + beta * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)
    idata = pm.sample(2000, tune=1000, chains=4, target_accept=0.9, random_seed=42)

print(az.summary(idata, var_names=["alpha", "beta", "sigma"]))
az.plot_trace(idata, var_names=["alpha", "beta"])

# Posterior predictive check: validación bayesiana fundamental
with modelo:
    ppc = pm.sample_posterior_predictive(idata, random_seed=42)
az.plot_ppc(ppc)

## 3. Modelo jerárquico (efectos por grupo)

`tip ~ Normal(α_día + β·total_bill, σ)` con `α_día ~ Normal(μ_α, σ_α)`. La jerarquía comparte información entre grupos (partial pooling), mejor que 4 regresiones separadas. Código correcto, **no ejecutado**.

In [ ]:
# Modelo jerárquico en PyMC v5 (ilustrativo, NO se ejecuta aquí)
import numpy as np
import pymc as pm
import arviz as az

rng = np.random.default_rng(0)
dia_idx = rng.integers(0, 4, 244)              # 4 días
total_bill = rng.gamma(4, 5, 244)
tip = 1.0 + 0.15 * total_bill + rng.normal(0, 1, 244)

with pm.Model() as jer:
    mu_a = pm.Normal("mu_a", 0, 5)
    sigma_a = pm.HalfNormal("sigma_a", 2)
    alpha_dia = pm.Normal("alpha_dia", mu_a, sigma_a, shape=4)   # intercepto por día
    beta = pm.Normal("beta", 0, 1)
    sigma = pm.HalfNormal("sigma", 2)
    mu = alpha_dia[dia_idx] + beta * total_bill
    pm.Normal("tip_obs", mu=mu, sigma=sigma, observed=tip)
    idata_jer = pm.sample(2000, tune=1000, chains=4, random_seed=0)

print(az.summary(idata_jer, var_names=["beta", "alpha_dia"]))
# HDI de β: si no incluye 0, el efecto del bill sobre la propina es claro
print(az.hdi(idata_jer, var_names=["beta"], hdi_prob=0.94))

## Ejercicios

1. En el conjugado, cambiá el prior a `Beta(2, 20)` (creencia previa de tasa baja) y observá cómo se desplaza el posterior con los mismos datos.
2. Ejecutá (en un entorno con PyMC) la regresión del bloque 2 y compará `mean` del posterior de `beta` contra el coeficiente OLS de `statsmodels`: con priors débiles deben coincidir.
3. Hacé un `pm.sample_prior_predictive` y verificá que los priors no generan datos absurdos (prior predictive check).

## Conclusiones

- El teorema de Bayes combina prior + likelihood en un posterior; en casos conjugados (Beta-Binomial) es analítico.
- El **HDI** sí tiene interpretación directa de probabilidad, a diferencia del IC frecuentista.
- PyMC v5 (NUTS) muestrea posteriores complejos; diagnosticá con `r_hat ≤ 1.01` y `ess_bulk ≥ 400`.
- El posterior predictive check es la validación bayesiana clave; los modelos jerárquicos comparten información entre grupos.